# 🛒 E-Commerce Data Cleaning Pipeline

End-to-end pipeline: raw data ingestion, profiling, cleaning, and preparation for SQL analysis.

**Data source:** UCI Machine Learning Repository — Online Retail Dataset (541,909 raw transactions)

## 1. Data Ingestion
Downloading the raw dataset directly from its public source and saving a local raw copy.

In [3]:
import pandas as pd

# 1. تحميل الداتاسيت الحقيقية الضخمة مباشرة من الإنترنت
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

print("⏳ جاري تحميل البيانات الحقيقية (أكثر من 500 ألف صف)... برجاء الانتظار ثواني...")
df_raw = pd.read_excel(url)

# 2. حفظ نسخة خام CSV على جهازك للمشروع
df_raw.to_csv("real_raw_ecommerce_2026.csv", index=False)

print(f"✅ تم تحميل وتصدير البيانات الحقيقية بنجاح! إجمالي الصفوف: {len(df_raw):,}")

⏳ جاري تحميل البيانات الحقيقية (أكثر من 500 ألف صف)... برجاء الانتظار ثواني...
✅ تم تحميل وتصدير البيانات الحقيقية بنجاح! إجمالي الصفوف: 541,909


## 2. Data Profiling
Inspecting the raw data structure, missing values, duplicates, and statistical anomalies (negative quantities/prices) before any cleaning.

In [4]:
import pandas as pd

# قراءة الملف الخام
df = pd.read_csv("real_raw_ecommerce_2026.csv")

# 1. معاينة أول 5 صفوف
print("--- 1. معاينة أول 5 صفوف ---")
display(df.head())

# 2. ملخص نوع البيانات والـ NULLs
print("\n--- 2. معلومات الأعمدة ونوع البيانات ---")
print(df.info())

# 3. حساب القيم المفقودة (Missing Values) بالظبط
print("\n--- 3. إجمالي القيم المفقودة في كل عمود ---")
print(df.isnull().sum())

# 4. حساب الصفوف المكررة (Duplicates)
print(f"\n--- 4. إجمالي الصفوف المكررة بالكامل: {df.duplicated().sum():,} ---")

# 5. ملخص إحصائي لكشف القيم السلبية أو الشاذة (Outliers / Negative Values)
print("\n--- 5. ملخص بالأرقام (الكميات والأسعار) ---")
display(df.describe())

--- 1. معاينة أول 5 صفوف ---


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



--- 2. معلومات الأعمدة ونوع البيانات ---
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 69.3 MB
None

--- 3. إجمالي القيم المفقودة في كل عمود ---
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

--- 4. إجمالي الصفوف المكررة بالكامل: 5,268 ---

--- 5. ملخص بالأرقام (الكميات والأسعار) ---


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


## 3. Data Cleaning & Transformation
Removing duplicates and invalid rows, correcting data types, engineering the `Total_Amount` feature, and exporting a clean dataset ready for SQL.

In [5]:
import pandas as pd

# 1. قراءة البيانات الخام
df = pd.read_csv("real_raw_ecommerce_2026.csv")

print(f"📊 عدد الصفوف قبل التنظيف: {len(df):,}")

# 2. إزالة الصفوف المكررة بالكامل
df.drop_duplicates(inplace=True)

# 3. إزالة المعاملات التي لا تحتوي على CustomerID (لأننا نبني تحليلاً يستهدف سلوك العملاء)
df.dropna(subset=['CustomerID'], inplace=True)

# 4. إزالة القيم السلبية أو الصفرية من الكميات والأسعار (استبعاد المرتجعات والأخطاء)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# 5. تصحيح أنواع البيانات (Data Types)
df['CustomerID'] = df['CustomerID'].astype(int).astype(str) # تحويل كود العميل لنص بدون أرقام عشرية
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate']) # تحويل لتاريخ حقيقي

# 6. إنشاء عمود إجمالي المبيعات (Total_Amount)
df['Total_Amount'] = df['Quantity'] * df['UnitPrice']

# 7. تنظيف النصوص في وصف المنتج من المسافات الزائدة
df['Description'] = df['Description'].str.strip().str.upper()

print(f"✅ عدد الصفوف بعد التنظيف الاحترافي: {len(df):,}")
print(f"🗑️ تم استبعاد {541909 - len(df):,} صف فوضوي أو غير صالح للتحليل.")

# 8. تصدير البيانات النظيفة كملف جديد جاهز لمرحلة الـ SQL
df.to_csv("clean_ecommerce_sales_2026.csv", index=False)
print("🚀 تم حفظ الملف المنظف بنجاح باسم: 'clean_ecommerce_sales_2026.csv'")

📊 عدد الصفوف قبل التنظيف: 541,909
✅ عدد الصفوف بعد التنظيف الاحترافي: 392,692
🗑️ تم استبعاد 149,217 صف فوضوي أو غير صالح للتحليل.
🚀 تم حفظ الملف المنظف بنجاح باسم: 'clean_ecommerce_sales_2026.csv'


## 4. Loading into SQL
Uploading the cleaned dataset into a SQLite database, with a quick validation query run directly from Python.

In [6]:
import pandas as pd
import sqlite3

# 1. قراءة الملف المنظف
df_clean = pd.read_csv("clean_ecommerce_sales_2026.csv")

# 2. إنشاء اتصال بقاعدة بيانات SQL (سيتم إنشاء ملف قاعدة بيانات باسم ecommerce_db.db)
conn = sqlite3.connect("ecommerce_db.db")

# 3. رفع البيانات إلى جدول جديد داخل SQL باسم 'sales_data'
df_clean.to_sql("sales_data", conn, if_exists="replace", index=False)

print("✅ تم رفع 392,692 صف بنجاح إلى قاعدة بيانات SQL!")

# 4. التست والتحقق: كتابة أول كود SQL داخل Python للتأكد من عمل الجدول
query = """
SELECT Country, COUNT(DISTINCT CustomerID) AS Total_Customers, SUM(Total_Amount) AS Total_Revenue
FROM sales_data
GROUP BY Country
ORDER BY Total_Revenue DESC
LIMIT 5;
"""

df_sql_result = pd.read_sql_query(query, conn)
print("\n📊 أعلى 5 دول تحقيقاً للمبيعات من داخل قاعدة بيانات SQL:")
display(df_sql_result)

# إغلاق الاتصال بقاعدة البيانات
conn.close()

✅ تم رفع 392,692 صف بنجاح إلى قاعدة بيانات SQL!

📊 أعلى 5 دول تحقيقاً للمبيعات من داخل قاعدة بيانات SQL:


,Country,Total_Customers,Total_Revenue
0,United Kingdom,3920,7285024.644
1,Netherlands,9,285446.340
2,EIRE,3,265262.460
3,Germany,94,228678.400
4,France,87,208934.310


## 5. SQL Analytics
Advanced queries: top customers by spend, and top 3 products per country using `DENSE_RANK()` with a CTE.

In [1]:
import pandas as pd
import sqlite3

# الاتصال بقاعدة البيانات
conn = sqlite3.connect("ecommerce_db.db")

# 1. Query 1: أفضل 10 عملاء باستخدام Subquery & Aggregation
query_top_customers = """
SELECT 
    CustomerID,
    Country,
    COUNT(DISTINCT InvoiceNo) AS Total_Orders,
    ROUND(SUM(Total_Amount), 2) AS Total_Spent,
    ROUND(AVG(Total_Amount), 2) AS Avg_Order_Value
FROM sales_data
GROUP BY CustomerID
ORDER BY Total_Spent DESC
LIMIT 10;
"""

print("🏆 أعلى 10 عملاء تحقيقاً للمبيعات:")
df_top_customers = pd.read_sql_query(query_top_customers, conn)
display(df_top_customers)

# 2. Query 2: ترتيب المنتجات الأعلى مبيعاً في كل دولة باستخدام Window Functions (DENSE_RANK)
query_window_func = """
WITH RankedProducts AS (
    SELECT 
        Country,
        Description,
        ROUND(SUM(Total_Amount), 2) AS Product_Revenue,
        DENSE_RANK() OVER (PARTITION BY Country ORDER BY SUM(Total_Amount) DESC) AS Rank
    FROM sales_data
    GROUP BY Country, Description
)
SELECT Country, Description, Product_Revenue, Rank
FROM RankedProducts
WHERE Rank <= 3 AND Country IN ('United Kingdom', 'Netherlands', 'Germany')
ORDER BY Country, Rank;
"""

print("\n🥇 أرفع 3 منتجات مبيعاً في أكبر 3 دول (باستخدام DENSE_RANK):")
df_ranked_products = pd.read_sql_query(query_window_func, conn)
display(df_ranked_products)

conn.close()

🏆 أعلى 10 عملاء تحقيقاً للمبيعات:


,CustomerID,Country,Total_Orders,Total_Spent,Avg_Order_Value
0,14646,Netherlands,73,280206.02,134.97
1,18102,United Kingdom,60,259657.30,602.45
2,17450,United Kingdom,46,194390.79,578.54
3,16446,United Kingdom,2,168472.50,56157.50
4,14911,EIRE,201,143711.17,25.35
5,12415,Australia,21,124914.53,174.95
6,14156,EIRE,55,117210.08,84.02
7,17511,United Kingdom,31,91062.38,94.56
8,16029,United Kingdom,63,80850.84,335.48
9,12346,United Kingdom,1,77183.60,77183.60



🥇 أرفع 3 منتجات مبيعاً في أكبر 3 دول (باستخدام DENSE_RANK):


,Country,Description,Product_Revenue,Rank
0,Germany,POSTAGE,21001.00,1
1,Germany,REGENCY CAKESTAND 3 TIER,9061.95,2
2,Germany,ROUND SNACK BOXES SET OF4 WOODLAND,3563.55,3
3,Netherlands,RABBIT NIGHT LIGHT,9568.48,1
4,Netherlands,ROUND SNACK BOXES SET OF4 WOODLAND,7991.40,2
5,Netherlands,SPACEBOY LUNCH BOX,7485.60,3
6,United Kingdom,"PAPER CRAFT , LITTLE BIRDIE",168469.60,1
7,United Kingdom,REGENCY CAKESTAND 3 TIER,110713.00,2
8,United Kingdom,WHITE HANGING HEART T-LIGHT HOLDER,94805.50,3
